# Scrap notebooks

This notebook is just a compilation of scrapy tests to view functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp

In [ ]:
%matplotlib widget

In [ ]:
phi = 0.3
phi_min = 0.1
phi_max = 0.9

alpha = 2
beta = 1
gamma = 4
delta = beta * gamma / (beta + gamma)

f = 0.5
f_min = 0
f_max = 10

E = 0.3
E_min = 0
E_max = 10

N = 10000
N_min = 1
N_max = 100000

s = 5000
s_min = 2000
s_max = 10000

M = 3
M_min = 1
M_max = 8

Delta_d = 1
Delta_d_min = -5
Delta_d_max = 15

Delta_p_min = 3
Delta_p_min_min = -2
Delta_p_min_max = 10
Delta_p_max = 5
Delta_p_max_min = Delta_p_min
Delta_p_max_max = 15
# F_generic = sp.stats.uniform(loc=Delta_p_min, scale=Delta_p_max - Delta_p_min)
F_generic = sp.stats.triang(c=0.5, loc=Delta_p_min, scale=Delta_p_max - Delta_p_min)
# F_generic = sp.stats.trapezoid(c=0.4, d=0.6, loc=Delta_p_min, scale=Delta_p_max - Delta_p_min)

In [ ]:
dim = 1000

# first depending variables
N_ds = np.linspace(-N, N, dim)
N_ps = np.linspace(0, N, dim)

N_ds, N_ps = np.meshgrid(N_ds, N_ps)
N_ss = N - N_ds - N_ps

mask = (N_ds >= 0) & (N_ps >= 0) & (N_ss >= 0) & (N_ps <= M * N_ds)

surface = np.zeros((dim, dim), dtype="float")

surface[~mask] = np.nan

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(N_ds, N_ps, surface, cmap=plt.cm.viridis, linewidth=0.2)

ax.set_xlabel('N_d')
ax.set_ylabel('N_p')


In [ ]:
integral_Delta_p_values = np.linspace(Delta_p_min, F_generic.ppf(N_ps[:, 0] / N), 10000, axis=1)
integral_Delta_p_deltas = np.diff(integral_Delta_p_values, axis=1)
integral_Delta_p_values = integral_Delta_p_values[:, 1:]
integral_Delta_p_Fvalues = F_generic.pdf(integral_Delta_p_values)
integral = (integral_Delta_p_Fvalues\
    * integral_Delta_p_deltas\
    * integral_Delta_p_values).sum(axis=1)
integral = np.transpose(np.tile(integral, (1000, 1)))

SC = (delta / s) * ((N_ss)**2/(1-phi) + (N_ds**2 + N_ds * N_ps)/phi)\
    + (N_ss + N_ds) * (f + E) + N_ds * Delta_d# + N*integral

In [ ]:
fig = plt.figure()

ax = fig.add_subplot(projection='3d')
ax.plot_surface(N_ds, N_ps, SC, cmap=plt.cm.viridis, linewidth=0.2)
ax.set_xlabel('N_d')
ax.set_ylabel('N_p')

In [ ]:
N_ps_1dim = N_ps[:, 0]
lhs = (f + E + delta * N / s + Delta_d * (1 + phi) / 2) * np.ones(dim)
rhs = delta * N_ps_1dim / ((1-phi) * s)  * (2 - 1/2 * (1+phi)*(1+phi) / phi) + F_generic.ppf(N_ps_1dim / N)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 5), sharex=True, tight_layout=True)
ax1.plot(N_ps_1dim, lhs, label="lhs")
ax1.plot(N_ps_1dim, rhs, label="rhs")

N_ds_1dim_of_N_ps = - N_ps_1dim / 2 * (1+phi) + phi*(1-phi) * (N / (1-phi) - Delta_d * s / (2 * delta))
ax2.plot(N_ps_1dim, N_ds_1dim_of_N_ps, label="N_d")

ax2.set_xlabel('N_p')
ax2.set_ylabel('N_d')

In [ ]:
fig, ax = plt.subplots()
ax.plot(N_ps[:, 0], SC[:, 0])


In [ ]:
N_p_1dim = N_ps[:, 0]
SC_1dim_N_d_0 = SC[:, 0]

N_p_1dim_diff = np.diff(N_p_1dim)
SC_1dim_N_d_0_diff = np.diff(SC_1dim_N_d_0) / N_p_1dim_diff

fig, ax = plt.subplots()
ax.plot(N_p_1dim[:-1], SC_1dim_N_d_0_diff)

SC_1dim_N_d_0_diff_me = delta / s * (-2*(N - N_p_1dim) / (1-phi)) - f - E + F_generic.ppf(N_p_1dim / N)
ax.plot(N_p_1dim, SC_1dim_N_d_0_diff_me, label="me")

In [ ]:
fig, ax = plt.subplots()
ax.plot(N_ps_1dim, integral[:, 0])

In [ ]:
integral_diff = np.diff(integral[:, 0]) / N_p_1dim_diff

fig, ax = plt.subplots()
ax.plot(N_ps_1dim[:-1], integral_diff)
integral_diff_me = F_generic.ppf(N_ps_1dim / N) / N
ax.plot(N_ps_1dim, integral_diff_me, label="me")


In [ ]:
x = np.linspace(0,1, 1000000)
dx = np.diff(x)
inte = (dx * (x[1:] + x[:-1])/2).sum()

print(inte)